In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# FinBERT Tone
tone_model_name = "yiyanghkust/finbert-tone"
tone_tokenizer = AutoTokenizer.from_pretrained(tone_model_name)
tone_model = AutoModelForSequenceClassification.from_pretrained(tone_model_name)
tone_classifier = pipeline(
    "text-classification",
    model=tone_model,
    tokenizer=tone_tokenizer,
    truncation=True,
    max_length=512,
    return_all_scores=True,
    top_k=None,
    device=-1,
 )

# FinBERT ESG
esg_model_name = "yiyanghkust/finbert-esg"
esg_tokenizer = AutoTokenizer.from_pretrained(esg_model_name)
esg_model = AutoModelForSequenceClassification.from_pretrained(esg_model_name)
esg_classifier = pipeline(
    "text-classification",
    model=esg_model,
    tokenizer=esg_tokenizer,
    truncation=True,
    max_length=512,
    return_all_scores=True,
    top_k=None,
    device=-1,
 )

print("Loaded FinBERT Tone and FinBERT ESG models.")

config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

C:\Users\wongb\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\wongb\.cache\huggingface\hub\models--yiyanghkust--finbert-tone. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

Device set to use cpu
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

Device set to use cpu


Loaded FinBERT Tone and FinBERT ESG models.


In [3]:
import re
import csv
from pathlib import Path
from collections import defaultdict

# Configuration
filings_root = Path("sec_filings")

def extract_year_from_filename(filename: str) -> int | None:
    """Extract year (2017-2025) from filename."""
    match = re.search(r'_10K_(\d{4})', filename)
    if match:
        year = int(match.group(1))
        if 2017 <= year <= 2025:
            return year
    return None

def find_risk_factor_headers(text: str) -> list[int]:
    """
    Find line numbers containing Risk Factors header.
    Criteria: Line must contain 'ITEM 1A' (or 'ITEM 1. A.') and 'Risk Factors' 
    and no other alphanumeric characters (punctuation/whitespace OK).
    """
    lines = text.split('\n')
    matching_lines = []
    
    for line_num, line in enumerate(lines, start=1):
        line_upper = line.upper()
        
        # Must contain "ITEM" and "1A" or "1" and "A" separately
        has_item = 'ITEM' in line_upper
        has_1a = '1A' in line_upper or ('1' in line_upper and 'A' in line_upper)
        has_risk_factor = 'RISK' in line_upper and 'FACTOR' in line_upper
        
        if not (has_item and has_1a and has_risk_factor):
            continue
        
        # Remove all non-alphanumeric except spaces, then check components
        # Keep only letters and digits for validation
        alphanumeric_only = re.sub(r'[^A-Za-z0-9\s]', ' ', line_upper)
        tokens = alphanumeric_only.split()
        
        # Valid tokens are only: ITEM, 1, A, 1A, RISK, FACTORS (and variations)
        valid_tokens = {'ITEM', '1', 'A', '1A', 'RISK', 'RISKS', 'FACTOR', 'FACTORS', 'ITEM1A'}
        
        # Check if all tokens are valid
        if tokens and all(token in valid_tokens for token in tokens):
            # Ensure we have the key components
            has_item_1a = ('ITEM' in tokens or 'ITEM1A' in tokens) and ('1A' in tokens or ('1' in tokens and 'A' in tokens) or 'ITEM1A' in tokens)
            has_risk_factors = any(t in tokens for t in ['RISK', 'RISKS']) and any(t in tokens for t in ['FACTOR', 'FACTORS'])
            
            if has_item_1a and has_risk_factors:
                matching_lines.append(line_num)
    
    return matching_lines

def find_unresolved_staff_comments_headers(text: str) -> list[int]:
    """
    Find line numbers containing ITEM 1B. UNRESOLVED STAFF COMMENTS header variants.
    Criteria: Line must contain ITEM, 1B (or 1 + B), UNRESOLVED, STAFF, and COMMENT(S),
    and no other alphanumeric tokens (punctuation/whitespace OK).
    """
    lines = text.split('\n')
    matching_lines = []

    for line_num, line in enumerate(lines, start=1):
        line_upper = line.upper()

        has_item = 'ITEM' in line_upper
        has_1b = '1B' in line_upper or ('1' in line_upper and 'B' in line_upper)
        has_unresolved = 'UNRESOLVED' in line_upper
        has_staff = 'STAFF' in line_upper
        has_comment = 'COMMENT' in line_upper

        if not (has_item and has_1b and has_unresolved and has_staff and has_comment):
            continue

        alphanumeric_only = re.sub(r'[^A-Za-z0-9\s]', ' ', line_upper)
        tokens = alphanumeric_only.split()

        valid_tokens = {
            'ITEM', '1', 'B', '1B', 'ITEM1B',
            'UNRESOLVED', 'STAFF', 'COMMENT', 'COMMENTS'
        }

        if tokens and all(token in valid_tokens for token in tokens):
            has_item_1b = (
                ('ITEM' in tokens or 'ITEM1B' in tokens)
                and ('1B' in tokens or ('1' in tokens and 'B' in tokens) or 'ITEM1B' in tokens)
            )
            has_unresolved_staff_comments = (
                'UNRESOLVED' in tokens
                and 'STAFF' in tokens
                and any(t in tokens for t in ['COMMENT', 'COMMENTS'])
            )

            if has_item_1b and has_unresolved_staff_comments:
                matching_lines.append(line_num)

    return matching_lines

# Process each ticker
for ticker_dir in sorted(filings_root.iterdir()):
    if not ticker_dir.is_dir():
        continue
    
    ticker = ticker_dir.name
    results = []
    
    # Find all 10K files
    for filing_file in sorted(ticker_dir.glob("*_10K_*.txt")):
        year = extract_year_from_filename(filing_file.name)
        if year is None:
            continue
        
        # Read file
        try:
            content = filing_file.read_text(encoding='utf-8', errors='ignore')
        except Exception as e:
            print(f"Error reading {filing_file.name}: {e}")
            results.append({
                'year': year,
                'match_count': 0,
                'line_numbers': '[]',
                'unresolved_staff_comments_line_numbers': '[]'
            })
            continue
        
        # Find headers
        line_numbers = find_risk_factor_headers(content)
        unresolved_staff_comments_line_numbers = find_unresolved_staff_comments_headers(content)
        
        results.append({
            'year': year,
            'match_count': len(line_numbers),
            'line_numbers': str(line_numbers),
            'unresolved_staff_comments_line_numbers': str(unresolved_staff_comments_line_numbers)
        })
    
    # Write CSV for this ticker in its own folder
    if results:
        csv_path = ticker_dir / f"{ticker}_risk_factors_headers.csv"
        with open(csv_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(
                f,
                fieldnames=['year', 'match_count', 'line_numbers', 'unresolved_staff_comments_line_numbers']
            )
            writer.writeheader()
            writer.writerows(results)
        print(f"Processed {ticker}: {len(results)} filings")

print("\nCompleted processing all tickers.")

Processed ADP: 9 filings
Processed ALLE: 8 filings
Processed AME: 8 filings
Processed AOS: 9 filings
Processed AXON: 8 filings
Processed BA: 9 filings
Processed BLDR: 8 filings
Processed BR: 9 filings
Processed CARR: 6 filings
Processed CAT: 9 filings
Processed CHRW: 9 filings
Processed CMI: 9 filings
Processed CPRT: 9 filings
Processed CSX: 9 filings
Processed CTAS: 9 filings
Processed DAL: 9 filings
Processed DE: 9 filings
Processed DOV: 9 filings
Processed EFX: 8 filings
Processed EME: 8 filings
Processed EMR: 9 filings
Processed ETN: 8 filings
Processed EXPD: 8 filings
Processed FAST: 9 filings
Processed FDX: 9 filings
Processed FIX: 8 filings
Processed FTV: 8 filings
Processed GD: 9 filings
Processed GE: 9 filings
Processed GEV: 2 filings
Processed GNRC: 8 filings
Processed GWW: 8 filings
Processed HII: 9 filings
Processed HON: 8 filings
Processed HUBB: 9 filings
Processed HWM: 9 filings
Processed IEX: 8 filings
Processed IR: 8 filings
Processed ITW: 9 filings
Processed J: 9 filin

In [ ]:
import ast
import pandas as pd
from pathlib import Path

# Configuration
filings_root = Path("sec_filings")

def parse_line_numbers(value) -> list[int]:
    """Parse a stored list-like line-number value into a list of ints."""
    if pd.isna(value):
        return []

    if isinstance(value, list):
        return [int(v) for v in value if isinstance(v, (int, float))]

    text = str(value).strip()
    if not text:
        return []

    try:
        parsed = ast.literal_eval(text)
    except Exception:
        return []

    if isinstance(parsed, list):
        return [int(v) for v in parsed if isinstance(v, (int, float))]

    return []

def row_is_errorless(row: pd.Series) -> bool:
    """
    A filing is errorless only if:
    - exactly one ITEM 1A Risk Factors header
    - exactly one ITEM 1B Unresolved Staff Comments header
    - ITEM 1B header appears after ITEM 1A header
    """
    risk_lines = parse_line_numbers(row.get('line_numbers'))
    unresolved_lines = parse_line_numbers(row.get('unresolved_staff_comments_line_numbers'))

    if len(risk_lines) != 1 or len(unresolved_lines) != 1:
        return False

    return unresolved_lines[0] > risk_lines[0]

def has_consecutive_errorless_years(df: pd.DataFrame, min_streak: int = 5) -> bool:
    """Return True if ticker has at least `min_streak` consecutive errorless years."""
    required_columns = {'year', 'line_numbers', 'unresolved_staff_comments_line_numbers'}
    if not required_columns.issubset(df.columns):
        return False

    work = df[list(required_columns)].dropna(subset=['year']).copy()
    work['year'] = pd.to_numeric(work['year'], errors='coerce')
    work = work.dropna(subset=['year'])

    if work.empty:
        return False

    work['year'] = work['year'].astype(int)
    work['is_errorless'] = work.apply(row_is_errorless, axis=1)

    errorless_years = sorted(set(work.loc[work['is_errorless'], 'year'].tolist()))

    if len(errorless_years) < min_streak:
        return False

    streak = 1
    for prev_year, year in zip(errorless_years, errorless_years[1:]):
        if year == prev_year + 1:
            streak += 1
            if streak >= min_streak:
                return True
        else:
            streak = 1

    return False

def has_errorless_year_range(df: pd.DataFrame, start_year: int, end_year: int) -> bool:
    """Return True if ticker has ALL years in range [start_year, end_year] as errorless."""
    required_columns = {'year', 'line_numbers', 'unresolved_staff_comments_line_numbers'}
    if not required_columns.issubset(df.columns):
        return False

    work = df[list(required_columns)].dropna(subset=['year']).copy()
    work['year'] = pd.to_numeric(work['year'], errors='coerce')
    work = work.dropna(subset=['year'])

    if work.empty:
        return False

    work['year'] = work['year'].astype(int)
    work['is_errorless'] = work.apply(row_is_errorless, axis=1)

    errorless_years = set(work.loc[work['is_errorless'], 'year'].tolist())
    required_years = set(range(start_year, end_year + 1))

    return required_years.issubset(errorless_years)

def split_tickers_by_error_count():
    """
    Split tickers into:
    - anomaly count > 2
    - anomaly count <= 2
    - duplicate year entries (can overlap with either list above)
    - at least 5 consecutive errorless years (can overlap with any list above)

    Error definition per 10-K filing:
    - not exactly one ITEM 1A Risk Factors header, OR
    - not exactly one ITEM 1B Unresolved Staff Comments header, OR
    - ITEM 1B not after ITEM 1A.
    """
    more_than_2_errors = []
    two_or_less_errors = []
    duplicate_year_tickers = []
    five_consecutive_errorless_tickers = []

    for ticker_dir in sorted(filings_root.iterdir()):
        if not ticker_dir.is_dir():
            continue

        ticker = ticker_dir.name
        csv_path = ticker_dir / f"{ticker}_risk_factors_headers.csv"

        if not csv_path.exists():
            continue

        df = pd.read_csv(csv_path)

        if 'line_numbers' not in df.columns:
            df['line_numbers'] = '[]'
        if 'unresolved_staff_comments_line_numbers' not in df.columns:
            df['unresolved_staff_comments_line_numbers'] = '[]'

        error_count = int((~df.apply(row_is_errorless, axis=1)).sum())

        if error_count > 2:
            more_than_2_errors.append(ticker)
        else:
            two_or_less_errors.append(ticker)

        if 'year' in df.columns and df['year'].duplicated().any():
            duplicate_year_tickers.append(ticker)

        if has_consecutive_errorless_years(df, min_streak=5):
            five_consecutive_errorless_tickers.append(ticker)

    return (
        more_than_2_errors,
        two_or_less_errors,
        duplicate_year_tickers,
        five_consecutive_errorless_tickers,
    )

# Run split
(
    more_than_2_errors,
    two_or_less_errors,
    duplicate_year_tickers,
    five_consecutive_errorless_tickers,
) = split_tickers_by_error_count()

print(f"Tickers with more than 2 errors (n={len(more_than_2_errors)}):")
print(more_than_2_errors)

print(f"\nTickers with 2 or fewer errors (n={len(two_or_less_errors)}):")
print(two_or_less_errors)

print(f"\nTickers with duplicate years (overlap allowed) (n={len(duplicate_year_tickers)}):")
print(duplicate_year_tickers)

print(f"\nTickers with at least 5 consecutive errorless years (overlap allowed) (n={len(five_consecutive_errorless_tickers)}):")
print(five_consecutive_errorless_tickers)

# Additional filters for year ranges
tickers_with_2020_2025 = []
tickers_with_2019_2025 = []
tickers_with_2017_2021 = []
tickers_with_2017_2025 = []

for ticker in five_consecutive_errorless_tickers:
    ticker_dir = filings_root / ticker
    csv_path = ticker_dir / f"{ticker}_risk_factors_headers.csv"
    
    if not csv_path.exists():
        continue
    
    df = pd.read_csv(csv_path)
    
    if has_errorless_year_range(df, 2020, 2025):
        tickers_with_2020_2025.append(ticker)
    
    if has_errorless_year_range(df, 2019, 2025):
        tickers_with_2019_2025.append(ticker)
    
    if has_errorless_year_range(df, 2017, 2021):
        tickers_with_2017_2021.append(ticker)
    
    if has_errorless_year_range(df, 2017, 2025):
        tickers_with_2017_2025.append(ticker)

print(f"\n\nTickers with 5+ consecutive errorless years AND all years 2020-2025 errorless (n={len(tickers_with_2020_2025)}):")
print(tickers_with_2020_2025)

print(f"\nTickers with 5+ consecutive errorless years AND all years 2019-2025 errorless (n={len(tickers_with_2019_2025)}):")
print(tickers_with_2019_2025)

# Show which tickers have 2020-2025 but NOT 2019-2025 (missing or errored 2019)
in_2020_not_2019 = set(tickers_with_2020_2025) - set(tickers_with_2019_2025)
print(f"\nTickers with 2020-2025 errorless but NOT 2019-2025 (missing or errored 2019):")
print(f"  {list(in_2020_not_2019)}")

print(f"\nTickers with 5+ consecutive errorless years AND all years 2017-2021 errorless (n={len(tickers_with_2017_2021)}):")
print(tickers_with_2017_2021)

print(f"\nTickers with 5+ consecutive errorless years AND all years 2017-2025 errorless (n={len(tickers_with_2017_2025)}):")
print(tickers_with_2017_2025)

Tickers with more than 2 errors (n=12):
['DAL', 'DE', 'EXPD', 'FDX', 'GE', 'HON', 'HUBB', 'ODFL', 'PAYC', 'PAYX', 'PH', 'ROL']

Tickers with 2 or fewer errors (n=67):
['ADP', 'ALLE', 'AME', 'AOS', 'AXON', 'BA', 'BLDR', 'BR', 'CARR', 'CAT', 'CHRW', 'CMI', 'CPRT', 'CSX', 'CTAS', 'DOV', 'EFX', 'EME', 'EMR', 'ETN', 'FAST', 'FIX', 'FTV', 'GD', 'GEV', 'GNRC', 'GWW', 'HII', 'HWM', 'IEX', 'IR', 'ITW', 'J', 'JBHT', 'JCI', 'LDOS', 'LHX', 'LII', 'LMT', 'LUV', 'MAS', 'MMM', 'NDSN', 'NOC', 'NSC', 'OTIS', 'PCAR', 'PNR', 'PWR', 'ROK', 'RSG', 'RTX', 'SNA', 'SWK', 'TDG', 'TT', 'TXT', 'UAL', 'UBER', 'UNP', 'UPS', 'URI', 'VLTO', 'VRSK', 'WAB', 'WM', 'XYL']

Tickers with duplicate years (overlap allowed) (n=0):
[]

Tickers with at least 5 consecutive errorless years (overlap allowed) (n=61):
['ADP', 'ALLE', 'AME', 'AXON', 'BA', 'BR', 'CARR', 'CAT', 'CHRW', 'CMI', 'CPRT', 'CSX', 'CTAS', 'DOV', 'EFX', 'EME', 'EMR', 'ETN', 'EXPD', 'FDX', 'FIX', 'FTV', 'GD', 'GNRC', 'GWW', 'HII', 'HWM', 'IEX', 'IR', 'ITW', 'J

In [10]:
import ast
import re
import pandas as pd
from pathlib import Path

# Paths
filings_root = Path("sec_filings")
output_root = Path("outputs")
sections_text_dir = output_root / "risk_sections_text"
sections_text_dir.mkdir(parents=True, exist_ok=True)

# Output files
master_spans_csv = output_root / "master_risk_sections.csv"
master_sentences_csv = output_root / "master_risk_sentences.csv"
skipped_rows_csv = output_root / "master_risk_sections_skipped.csv"

# Use only tickers that already passed the 5-consecutive-errorless filter in Cell 3
if 'five_consecutive_errorless_tickers' not in globals():
    raise RuntimeError(
        "`tickers_with_2019_2025` not found. Run Cell 3 first, then run Cell 4."
    )

target_tickers = set(tickers_with_2019_2025)
if not target_tickers:
    raise RuntimeError(
        "`five_consecutive_errorless_tickers` is empty. Re-run Cell 3 and confirm results before Cell 4."
    )


def parse_line_numbers(value) -> list[int]:
    """Parse stored list-like line number values (e.g., '[123]' or '[]')."""
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
    except Exception:
        return []
    if isinstance(parsed, list):
        return [int(v) for v in parsed if isinstance(v, (int, float))]
    return []


def split_into_sentences(text: str) -> list[str]:
    """
    Lightweight sentence splitter for FinBERT prep.
    Keeps sentence boundaries by splitting on punctuation + whitespace.
    """
    normalized = re.sub(r"\s+", " ", text).strip()
    if not normalized:
        return []

    # Split when punctuation likely ends a sentence, then clean empties.
    parts = re.split(r"(?<=[.!?])\s+", normalized)
    sentences = [p.strip() for p in parts if p and p.strip()]
    return sentences


def get_filing_path(ticker: str, year: int, ticker_dir: Path) -> Path | None:
    """Find the filing txt path for a ticker/year."""
    candidates = sorted(ticker_dir.glob(f"{ticker}_10K_{year}_*.txt"))
    if not candidates:
        return None
    return candidates[0]


span_rows = []
sentence_rows = []
skipped_rows = []

for ticker_dir in sorted(filings_root.iterdir()):
    if not ticker_dir.is_dir():
        continue

    ticker = ticker_dir.name
    if ticker not in target_tickers:
        continue

    csv_path = ticker_dir / f"{ticker}_risk_factors_headers.csv"
    if not csv_path.exists():
        continue

    df = pd.read_csv(csv_path)

    # Need these fields to identify clean between-header spans.
    required = {'year', 'line_numbers', 'unresolved_staff_comments_line_numbers'}
    if not required.issubset(df.columns):
        skipped_rows.append({
            'ticker': ticker,
            'year': None,
            'reason': 'missing_required_columns'
        })
        continue

    for _, row in df.iterrows():
        year_raw = row.get('year')
        if pd.isna(year_raw):
            skipped_rows.append({'ticker': ticker, 'year': None, 'reason': 'missing_year'})
            continue

        year = int(year_raw)
        risk_lines = parse_line_numbers(row.get('line_numbers'))
        unresolved_lines = parse_line_numbers(row.get('unresolved_staff_comments_line_numbers'))

        # Strict condition: exactly one each, and ITEM 1B after ITEM 1A.
        if len(risk_lines) != 1 or len(unresolved_lines) != 1:
            skipped_rows.append({
                'ticker': ticker,
                'year': year,
                'reason': 'header_count_not_equal_to_one_each'
            })
            continue

        risk_line = risk_lines[0]
        unresolved_line = unresolved_lines[0]

        if unresolved_line <= risk_line:
            skipped_rows.append({
                'ticker': ticker,
                'year': year,
                'reason': 'unresolved_not_after_risk'
            })
            continue

        filing_path = get_filing_path(ticker, year, ticker_dir)
        if filing_path is None:
            skipped_rows.append({
                'ticker': ticker,
                'year': year,
                'reason': 'filing_file_not_found'
            })
            continue

        try:
            filing_text = filing_path.read_text(encoding='utf-8', errors='ignore')
        except Exception:
            skipped_rows.append({
                'ticker': ticker,
                'year': year,
                'reason': 'filing_read_error'
            })
            continue

        lines = filing_text.splitlines()

        # Text strictly between headers (excluding header lines themselves).
        start_idx = risk_line
        end_idx_exclusive = unresolved_line - 1

        if start_idx >= len(lines) or end_idx_exclusive <= start_idx:
            skipped_rows.append({
                'ticker': ticker,
                'year': year,
                'reason': 'invalid_between_header_line_range'
            })
            continue

        between_text = "\n".join(lines[start_idx:end_idx_exclusive]).strip()
        if not between_text:
            skipped_rows.append({
                'ticker': ticker,
                'year': year,
                'reason': 'empty_between_header_text'
            })
            continue

        section_text_path = sections_text_dir / f"{ticker}_{year}_risk_to_unresolved.txt"
        section_text_path.write_text(between_text, encoding='utf-8')

        sentences = split_into_sentences(between_text)

        span_rows.append({
            'ticker': ticker,
            'year': year,
            'filing_path': str(filing_path),
            'risk_header_line': risk_line,
            'unresolved_header_line': unresolved_line,
            'section_text_path': str(section_text_path),
            'section_char_count': len(between_text),
            'sentence_count': len(sentences)
        })

        for idx, sentence in enumerate(sentences, start=1):
            sentence_rows.append({
                'ticker': ticker,
                'year': year,
                'filing_path': str(filing_path),
                'section_text_path': str(section_text_path),
                'sentence_id': idx,
                'total_sentences': len(sentences),
                'sentence_text': sentence
            })

# Save master datasets
spans_df = pd.DataFrame(span_rows)
sentences_df = pd.DataFrame(sentence_rows)
skipped_df = pd.DataFrame(skipped_rows)

spans_df.to_csv(master_spans_csv, index=False, encoding='utf-8')
sentences_df.to_csv(master_sentences_csv, index=False, encoding='utf-8')
skipped_df.to_csv(skipped_rows_csv, index=False, encoding='utf-8')

print(f"Target tickers from Cell 3: {len(target_tickers)}")
print(f"Saved span-level dataset: {master_spans_csv} (rows={len(spans_df)})")
print(f"Saved sentence-level dataset: {master_sentences_csv} (rows={len(sentences_df)})")
print(f"Saved skipped diagnostics: {skipped_rows_csv} (rows={len(skipped_df)})")
print(f"Saved extracted section text files in: {sections_text_dir}")

Target tickers from Cell 3: 28
Saved span-level dataset: outputs\master_risk_sections.csv (rows=250)
Saved sentence-level dataset: outputs\master_risk_sentences.csv (rows=76217)
Saved skipped diagnostics: outputs\master_risk_sections_skipped.csv (rows=0)
Saved extracted section text files in: outputs\risk_sections_text
